# Phishing URL Detection

## Objective

In this notebook, I develop a machine learning classifier for identifying phishing URLs using features extracted directly from URL strings.

My objective is to create a CPU-friendly detection component that can analyze suspicious links without visiting the corresponding websites.

I begin by exploring and cleaning the PhiUSIIL phishing URL dataset, then engineer lexical and structural URL features.

I compare suitable machine learning algorithms, evaluate their performance using leakage-aware dataset partitions, and save the selected model for integration with my existing text classifiers.

I use the same label convention as my previous notebooks:

- 0: Legitimate
- 1: Phishing

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

# I configure portable project paths.
PROJECT_ROOT = Path.cwd().resolve().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

URL_DATA_PATH = (
    RAW_DATA_DIR / "PhiUSIIL_Phishing_URL_Dataset.csv"
)

assert URL_DATA_PATH.is_file(), (
    f"URL dataset not found: {URL_DATA_PATH}"
)

print("Project root:", PROJECT_ROOT)
print("URL dataset:", URL_DATA_PATH.name)
print(
    "Dataset size:",
    f"{URL_DATA_PATH.stat().st_size / (1024 ** 2):.2f} MB"
)
print("Dataset verification: PASSED")

Project root: C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector
URL dataset: PhiUSIIL_Phishing_URL_Dataset.csv
Dataset size: 54.22 MB
Dataset verification: PASSED


## 1. Initial Dataset Exploration

I begin by examining the structure of the PhiUSIIL phishing URL dataset.

My objective is to identify the original URL strings, understand the target labels, and assess data quality before preprocessing.

Although the dataset contains precomputed URL and webpage features, I intend to build a detector using features extracted directly from URL strings.

This approach allows my classifier to analyze URLs without visiting potentially malicious websites during inference.

I first inspect the dataset dimensions, columns, missing values, class distribution, and duplicate URLs.

In [5]:
import time

start_time = time.perf_counter()

# I load the original URL dataset.
url_df = pd.read_csv(
    URL_DATA_PATH,
    low_memory=False
)

load_time = time.perf_counter() - start_time

print("URL DATASET OVERVIEW")
print("Shape:", url_df.shape)
print(f"Loading time: {load_time:.2f} seconds")

print("\nCOLUMN NAMES")
print(url_df.columns.tolist())

print("\nDATA TYPES")
print(url_df.dtypes.to_string())

print("\nFIRST FIVE RECORDS")
display(url_df.head())

URL DATASET OVERVIEW
Shape: (235795, 56)
Loading time: 1.19 seconds

COLUMN NAMES
['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [7]:
# I examine missing values across the dataset.
missing_summary = (
    url_df.isna()
    .sum()
    .sort_values(ascending=False)
)

print("MISSING VALUES")
print(missing_summary[missing_summary > 0].head(20))

# I inspect the expected target column.
if "label" in url_df.columns:
    print("\nLABEL DISTRIBUTION")
    print(url_df["label"].value_counts(dropna=False))
else:
    print("\nLabel column not found. Available columns:")
    print(url_df.columns.tolist())

# I inspect duplicate complete records.
print("\nEXACT DUPLICATE ROWS")
print(url_df.duplicated().sum())

# I inspect duplicate URL strings.
if "URL" in url_df.columns:
    print("\nDUPLICATE URL STRINGS")
    print(url_df["URL"].duplicated().sum())

    print("\nMISSING URL VALUES")
    print(url_df["URL"].isna().sum())

MISSING VALUES
Series([], dtype: int64)

LABEL DISTRIBUTION
label
1    134850
0    100945
Name: count, dtype: int64

EXACT DUPLICATE ROWS
0

DUPLICATE URL STRINGS
425

MISSING URL VALUES
0


## 2. URL Duplicate and Label Consistency Analysis

I investigate duplicate URL records before preparing the dataset for modeling.

Although the dataset contains no completely duplicated rows, some URL strings appear more than once.

I examine whether these repeated URLs share the same classification label or contain conflicting annotations.

This is important because identical URLs must not appear in different training, validation, and test partitions.

I also verify the original label convention before converting it to match my existing detection models.

In [10]:
# I identify URLs appearing more than once.
duplicate_url_mask = url_df["URL"].duplicated(
    keep=False
)

duplicate_url_records = url_df.loc[
    duplicate_url_mask,
    ["URL", "label"]
].copy()

print("DUPLICATE URL ANALYSIS")

print(
    "Total records belonging to duplicate groups:",
    len(duplicate_url_records)
)

print(
    "Unique duplicated URLs:",
    duplicate_url_records["URL"].nunique()
)

print(
    "Additional duplicate records:",
    url_df["URL"].duplicated().sum()
)

# I examine how frequently URLs are repeated.
url_frequency = url_df["URL"].value_counts()

print("\nMOST FREQUENT URL COUNTS")
print(url_frequency.head(10).to_string())

DUPLICATE URL ANALYSIS
Total records belonging to duplicate groups: 850
Unique duplicated URLs: 425
Additional duplicate records: 425

MOST FREQUENT URL COUNTS
URL
https://att-104164.weeblysite.com/                                                                                           2
https://etiliseramik.com/wp-includes/vector/login.htm                                                                        2
https://snapdragon-magic-flag.glitch.me/navyfederal.org.html                                                                 2
http://my-business-108580.square.site/                                                                                       2
https://facebook-help-600674518635606.firebaseapp.com/                                                                       2
http://splaudioprofessional.com/wp-content/netnet/                                                                           2
https://objectstorage.ap-singapore-1.oraclecloud.com/n/axk8tbjrgpni/b/3274

In [12]:
# I count distinct labels assigned to each URL.
url_label_counts = (
    url_df.groupby("URL")["label"]
    .nunique()
)

# I identify URLs appearing with conflicting classifications.
conflicting_urls = url_label_counts[
    url_label_counts > 1
]

print("URL LABEL CONSISTENCY")

print(
    "URLs with conflicting labels:",
    len(conflicting_urls)
)

if len(conflicting_urls) > 0:

    conflicting_records = url_df[
        url_df["URL"].isin(conflicting_urls.index)
    ]

    print(
        "Records affected:",
        len(conflicting_records)
    )

    print("\nConflicting label combinations:")

    display(
        conflicting_records.groupby("URL")["label"]
        .agg(["count", "nunique", "mean"])
        .head(10)
    )

else:
    print("No conflicting URL labels detected.")

URL LABEL CONSISTENCY
URLs with conflicting labels: 0
No conflicting URL labels detected.


## 3. URL Dataset Cleaning

I prepare a clean URL classification dataset by retaining the original URL strings and their corresponding labels.

I remove exact URL duplicates after confirming that repeated URLs have consistent classifications.

I also convert the original PhiUSIIL label convention to match my existing detection models:

- 0: Legitimate
- 1: Phishing

I preserve the original URL structure because its characters, hostname, path, and query parameters may contain useful information for identifying suspicious links.

I exclude the dataset's precomputed features so that my URL classifier can operate independently of webpage content.

In [15]:
# I retain only the original URL and classification label.
url_clean = url_df[["URL", "label"]].copy()

# I standardize column names.
url_clean = url_clean.rename(
    columns={"URL": "url"}
)

# I remove repeated URL records.
url_clean = url_clean.drop_duplicates(
    subset="url",
    keep="first"
).reset_index(drop=True)

# I convert the original label convention:
# Original: 0 = phishing, 1 = legitimate.
# Standard: 0 = legitimate, 1 = phishing.
url_clean["label"] = 1 - url_clean["label"]

print("CLEANED URL DATASET")

print("Original records:", len(url_df))
print("Cleaned records:", len(url_clean))
print("Records removed:", len(url_df) - len(url_clean))

print("\nLABEL DISTRIBUTION")
print(url_clean["label"].value_counts().sort_index())

print("\nDATASET PREVIEW")
display(url_clean.head())

CLEANED URL DATASET
Original records: 235795
Cleaned records: 235370
Records removed: 425

LABEL DISTRIBUTION
label
0    134850
1    100520
Name: count, dtype: int64

DATASET PREVIEW


,url,label
0,https://www.southbankmosaics.com,0
1,https://www.uni-mainz.de,0
2,https://www.voicefmradio.co.uk,0
3,https://www.sfnmjournal.com,0
4,https://www.rewildingargentina.org,0


In [17]:
# I verify the cleaned URL dataset.
assert len(url_clean) == 235370

assert url_clean["url"].is_unique

assert url_clean["url"].notna().all()

assert url_clean["label"].isin([0, 1]).all()

assert url_clean["url"].str.strip().ne("").all()

print("DATASET VALIDATION")
print("Unique URLs: PASSED")
print("Missing URLs: PASSED")
print("Binary labels: PASSED")
print("Empty URL strings: PASSED")

print("\nCleaned dataset shape:", url_clean.shape)

DATASET VALIDATION
Unique URLs: PASSED
Missing URLs: PASSED
Binary labels: PASSED
Empty URL strings: PASSED

Cleaned dataset shape: (235370, 2)


## 4. Hostname Analysis

I examine hostname repetition within the cleaned URL dataset before creating the training, validation, and test partitions.

Although I removed exact URL duplicates, different URLs may still belong to the same website.

I extract hostnames directly from the original URL strings and examine how frequently they occur.

I also investigate whether individual hostnames contain both legitimate and phishing-labeled URLs.

These findings will help me choose an appropriate group-aware splitting strategy and reduce the possibility of website-specific information leaking between dataset partitions.

In [20]:
from urllib.parse import urlsplit
import ipaddress

# I extract hostnames without accessing the websites.
def extract_hostname(url):
    try:
        url = str(url).strip()

        # I add a temporary scheme when none is present.
        if "://" not in url:
            url = "http://" + url

        hostname = urlsplit(url).hostname

        if hostname is None:
            return None

        # I standardize hostname casing and trailing dots.
        return hostname.lower().rstrip(".")

    except (ValueError, TypeError):
        return None


# I identify whether a hostname is an IP address.
def is_ip_hostname(hostname):
    if not hostname:
        return False

    try:
        ipaddress.ip_address(hostname)
        return True
    except ValueError:
        return False


# I create temporary hostname analysis columns.
url_host_analysis = url_clean.copy()

url_host_analysis["hostname"] = (
    url_host_analysis["url"].map(extract_hostname)
)

url_host_analysis["is_ip_host"] = (
    url_host_analysis["hostname"].map(is_ip_hostname)
)

print("HOSTNAME EXTRACTION RESULTS")
print("Total URLs:", len(url_host_analysis))

print(
    "Successfully extracted:",
    url_host_analysis["hostname"].notna().sum()
)

print(
    "Failed extractions:",
    url_host_analysis["hostname"].isna().sum()
)

print(
    "Unique hostnames:",
    url_host_analysis["hostname"].nunique()
)

print(
    "IP-based hostnames:",
    url_host_analysis["is_ip_host"].sum()
)

HOSTNAME EXTRACTION RESULTS
Total URLs: 235370
Successfully extracted: 235370
Failed extractions: 0
Unique hostnames: 220077
IP-based hostnames: 606


In [22]:
# I calculate how frequently each hostname appears.
hostname_counts = (
    url_host_analysis["hostname"]
    .value_counts()
)

repeated_hostnames = hostname_counts[
    hostname_counts > 1
]

print("HOSTNAME REPETITION ANALYSIS")

print(
    "Unique hostnames:",
    len(hostname_counts)
)

print(
    "Hostnames appearing more than once:",
    len(repeated_hostnames)
)

print(
    "URLs belonging to repeated hostnames:",
    int(repeated_hostnames.sum())
)

print(
    "Largest hostname group:",
    int(hostname_counts.max())
)

print("\nHOSTNAME GROUP SIZE DISTRIBUTION")

display(
    hostname_counts.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)

HOSTNAME REPETITION ANALYSIS
Unique hostnames: 220077
Hostnames appearing more than once: 5256
URLs belonging to repeated hostnames: 20549
Largest hostname group: 1192

HOSTNAME GROUP SIZE DISTRIBUTION


count    220077.000000
mean          1.069489
std           3.456445
min           1.000000
50%           1.000000
90%           1.000000
95%           1.000000
99%           2.000000
max        1192.000000
Name: count, dtype: float64

In [24]:
# I count distinct labels within each hostname.
hostname_label_counts = (
    url_host_analysis
    .dropna(subset=["hostname"])
    .groupby("hostname")["label"]
    .nunique()
)

mixed_label_hostnames = hostname_label_counts[
    hostname_label_counts > 1
]

print("HOSTNAME LABEL ANALYSIS")

print(
    "Hostnames containing both labels:",
    len(mixed_label_hostnames)
)

mixed_label_records = url_host_analysis[
    url_host_analysis["hostname"].isin(
        mixed_label_hostnames.index
    )
]

print(
    "URLs belonging to mixed-label hostnames:",
    len(mixed_label_records)
)

print("\nTOP 10 MOST FREQUENT HOSTNAMES")

hostname_summary = (
    url_host_analysis
    .dropna(subset=["hostname"])
    .groupby("hostname")
    .agg(
        url_count=("url", "size"),
        phishing_count=("label", "sum"),
        phishing_rate=("label", "mean")
    )
    .sort_values("url_count", ascending=False)
)

display(hostname_summary.head(10))

HOSTNAME LABEL ANALYSIS
Hostnames containing both labels: 54
URLs belonging to mixed-label hostnames: 112

TOP 10 MOST FREQUENT HOSTNAMES


,url_count,phishing_count,phishing_rate
hostname,,,
ipfs.io,1192,1192,1.0
docs.google.com,526,526,1.0
mail.deliverylifesupport.com,370,370,1.0
cloudflare-ipfs.com,359,359,1.0
storageapi.fleek.co,318,318,1.0
gateway.pinata.cloud,275,275,1.0
gateway.ipfs.io,249,249,1.0
ipfs.litnet.work,223,223,1.0
cf-ipfs.com,190,190,1.0


## 5. Hostname-Aware Dataset Splitting

I create training, validation, and test partitions using hostname-level grouping.

My initial exploration revealed that some hostnames appear repeatedly, including legitimate hosting platforms that contain phishing-labeled URLs.

To reduce evaluation leakage, I ensure that URLs sharing the same hostname remain in the same partition.

I target a 70/15/15 split and stratify hostname groups using their majority classification label.

I preserve the original classification of every individual URL, including URLs belonging to mixed-label hostname groups.

This approach prevents exact hostname overlap between partitions, although it does not eliminate all possible similarities between related domains or hosting infrastructure.

In [27]:
# Prepare hostname-level metadata
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# I summarize the number of URLs and phishing labels per hostname.
hostname_group_info = (
    url_host_analysis
    .groupby("hostname")
    .agg(
        group_size=("label", "size"),
        phishing_count=("label", "sum")
    )
    .reset_index()
)

# I calculate the number of legitimate URLs.
hostname_group_info["legitimate_count"] = (
    hostname_group_info["group_size"]
    - hostname_group_info["phishing_count"]
)

# I assign a majority-class stratum for splitting.
# This does not change individual URL labels.
hostname_group_info["stratum"] = (
    hostname_group_info["phishing_count"]
    >= hostname_group_info["legitimate_count"]
).astype("int8")

print("HOSTNAME GROUP METADATA")

print("Total hostname groups:", len(hostname_group_info))
print("Total URLs:", hostname_group_info["group_size"].sum())

print("\nGROUP STRATA")
print(
    hostname_group_info["stratum"]
    .value_counts()
    .sort_index()
)

print("\nGROUP SIZE STATISTICS")
print(
    hostname_group_info["group_size"]
    .describe()
)

HOSTNAME GROUP METADATA
Total hostname groups: 220077
Total URLs: 235370

GROUP STRATA
stratum
0    134796
1     85281
Name: count, dtype: int64

GROUP SIZE STATISTICS
count    220077.000000
mean          1.069489
std           3.456445
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max        1192.000000
Name: group_size, dtype: float64


In [29]:
# I assign approximately 70% of hostname groups to training.
url_train_groups, url_temp_groups = train_test_split(
    hostname_group_info,
    test_size=0.30,
    stratify=hostname_group_info["stratum"],
    random_state=RANDOM_STATE
)

# I divide the remaining groups equally between
# validation and testing.
url_val_groups, url_test_groups = train_test_split(
    url_temp_groups,
    test_size=0.50,
    stratify=url_temp_groups["stratum"],
    random_state=RANDOM_STATE
)

# I collect the hostname identifiers for each partition.
url_train_hosts = set(url_train_groups["hostname"])
url_val_hosts = set(url_val_groups["hostname"])
url_test_hosts = set(url_test_groups["hostname"])

print("HOSTNAME GROUP ALLOCATION")

print("Training groups:", len(url_train_hosts))
print("Validation groups:", len(url_val_hosts))
print("Test groups:", len(url_test_hosts))

HOSTNAME GROUP ALLOCATION
Training groups: 154053
Validation groups: 33012
Test groups: 33012


In [31]:
# I assign URLs according to their hostname groups.
url_train = url_host_analysis[
    url_host_analysis["hostname"].isin(url_train_hosts)
].copy()

url_val = url_host_analysis[
    url_host_analysis["hostname"].isin(url_val_hosts)
].copy()

url_test = url_host_analysis[
    url_host_analysis["hostname"].isin(url_test_hosts)
].copy()

# I retain the URL and its original standardized label.
url_partitions = {
    "train": url_train[["url", "label"]].reset_index(drop=True),
    "val": url_val[["url", "label"]].reset_index(drop=True),
    "test": url_test[["url", "label"]].reset_index(drop=True)
}

print("URL DATASET PARTITIONS")

for name, partition in url_partitions.items():

    print(
        f"{name.upper()}: "
        f"{len(partition):,} URLs | "
        f"Phishing: {partition['label'].mean():.2%}"
    )

URL DATASET PARTITIONS
TRAIN: 164,043 URLs | Phishing: 42.46%
VAL: 34,942 URLs | Phishing: 42.10%
TEST: 36,385 URLs | Phishing: 44.41%


In [33]:
# I verify that hostname groups do not overlap.
assert url_train_hosts.isdisjoint(url_val_hosts)
assert url_train_hosts.isdisjoint(url_test_hosts)
assert url_val_hosts.isdisjoint(url_test_hosts)

print("Hostname separation: PASSED")


# I verify that all URLs were assigned.
total_partition_records = sum(
    len(df) for df in url_partitions.values()
)

assert total_partition_records == len(url_clean)

# I verify that no URL appears in multiple partitions.
all_partition_urls = pd.concat(
    [
        df["url"]
        for df in url_partitions.values()
    ],
    ignore_index=True
)

assert all_partition_urls.is_unique

print("Record coverage: PASSED")
print("URL uniqueness: PASSED")


# I inspect the final split proportions and label distributions.
url_split_summary = pd.DataFrame([
    {
        "split": name,
        "records": len(df),
        "percentage": len(df) / len(url_clean) * 100,
        "phishing_rate": df["label"].mean() * 100
    }
    for name, df in url_partitions.items()
])

print("\nFINAL URL SPLIT SUMMARY")
display(url_split_summary.round(2))

Hostname separation: PASSED
Record coverage: PASSED
URL uniqueness: PASSED

FINAL URL SPLIT SUMMARY


,split,records,percentage,phishing_rate
0,train,164043,69.70,42.46
1,val,34942,14.85,42.10
2,test,36385,15.46,44.41


## 6. Final URL Dataset Partitions

I finalized my URL dataset partitions using hostname-aware splitting.

I kept all URLs sharing the same hostname within a single partition and verified that the resulting datasets preserve reasonable class distributions.

The final allocation contains approximately 70% training data, 15% validation data, and 15% test data.

I save the partitions as Parquet files for reproducible model development.

I reserve the test partition for final evaluation and avoid using it during feature engineering, model selection, or hyperparameter tuning.

In [36]:
# I create the output directory for processed dataset splits.
SPLIT_DIR = PROCESSED_DATA_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# I save the validated URL partitions.
for split_name, partition in url_partitions.items():

    output_path = SPLIT_DIR / f"url_{split_name}.parquet"

    partition.to_parquet(
        output_path,
        index=False
    )

    print(
        f"Saved: {output_path.name} "
        f"({len(partition):,} URLs)"
    )

Saved: url_train.parquet (164,043 URLs)
Saved: url_val.parquet (34,942 URLs)
Saved: url_test.parquet (36,385 URLs)


In [38]:
# I reload each saved partition to verify data integrity.
for split_name, expected_df in url_partitions.items():

    saved_path = SPLIT_DIR / f"url_{split_name}.parquet"

    reloaded_df = pd.read_parquet(saved_path)

    pd.testing.assert_frame_equal(
        reloaded_df,
        expected_df
    )

    print(f"{split_name.upper()}: VERIFIED")

TRAIN: VERIFIED
VAL: VERIFIED
TEST: VERIFIED


## 7. URL Feature Engineering

I develop a reusable feature extractor that converts URL strings into numerical representations.

The extractor uses lexical and structural characteristics such as URL length, hostname complexity, suspicious keywords, special characters, and HTTPS usage.

I implement the extractor separately from the notebook so that the same transformations can be applied during training, validation, testing, and application inference.

All features are calculated locally without visiting the corresponding websites.

In [41]:
import sys

# I make the project root available for local module imports.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.url_features import (
    extract_url_features,
    extract_features_dataframe
)

# I use synthetic URLs to test the extractor.
sample_urls = [
    "https://example.com",
    "https://accounts.example.com/login",
    "http://192.0.2.10/verify-account",
    "https://example.org/search?q=security",
    "https://example.net/%61ccount/update"
]

sample_features = extract_features_dataframe(sample_urls)

print("URL FEATURE EXTRACTOR TEST")

print("Sample URLs:", len(sample_urls))
print("Extracted features:", sample_features.shape[1])

display(sample_features)

# I verify the expected output dimensions.
assert sample_features.shape == (5, 21)

# I verify that the extractor produces no missing values.
assert not sample_features.isna().any().any()

# I verify the IP-based hostname feature.
assert sample_features.loc[2, "is_ip_host"] == 1

# I verify HTTPS detection.
assert sample_features.loc[0, "is_https"] == 1
assert sample_features.loc[2, "is_https"] == 0

print("\nFeature extraction validation: PASSED")

URL FEATURE EXTRACTOR TEST
Sample URLs: 5
Extracted features: 21


,url_length,hostname_length,path_length,query_length,digit_count,digit_ratio,special_char_count,dot_count,hyphen_count,slash_count,...,question_mark_count,equal_sign_count,percent_encoding_count,subdomain_count,path_depth,is_ip_host,is_https,has_query,has_fragment,suspicious_keyword_count
0,19,11,0,0,0,0.000000,4,1,0,2,...,0,0,0,0,0,0,1,0,0,0
1,34,20,6,0,0,0.000000,6,2,0,3,...,0,0,0,1,1,0,1,0,0,2
2,32,10,15,0,7,0.218750,8,3,1,3,...,0,0,0,2,1,1,0,0,0,2
3,37,11,7,10,0,0.000000,7,1,0,3,...,1,1,0,0,1,0,1,1,0,0
4,36,11,17,0,2,0.055556,7,1,0,4,...,0,0,1,0,2,0,1,0,0,1



Feature extraction validation: PASSED


In [43]:
import importlib
import src.url_features

importlib.reload(src.url_features)

from src.url_features import (
    extract_url_features,
    extract_features_dataframe
)

sample_features = extract_features_dataframe(sample_urls)

assert sample_features.loc[2, "subdomain_count"] == 0

print("IP hostname correction: PASSED")

IP hostname correction: PASSED


## 8. URL Feature Matrix Preparation

I apply my validated URL feature extractor to the training and validation partitions.

Each URL is represented using 21 numerical features derived entirely from its string structure.

I generate these features independently of the target labels and without accessing the corresponding websites.

I verify that both partitions contain consistent feature columns, valid numerical values, and the expected number of records.

I reserve the test partition for final evaluation.

In [48]:
# I load the fixed hostname-aware URL partitions.
url_train_df = pd.read_parquet(
    SPLIT_DIR / "url_train.parquet"
)

url_val_df = pd.read_parquet(
    SPLIT_DIR / "url_val.parquet"
)

print("URL TRAINING DATA")
print("Shape:", url_train_df.shape)
print("Phishing rate:", f"{url_train_df['label'].mean():.2%}")

print("\nURL VALIDATION DATA")
print("Shape:", url_val_df.shape)
print("Phishing rate:", f"{url_val_df['label'].mean():.2%}")

# I verify that the expected columns are available.
for name, df in [
    ("Training", url_train_df),
    ("Validation", url_val_df)
]:
    assert {"url", "label"}.issubset(df.columns)
    assert df["url"].notna().all()
    assert df["label"].isin([0, 1]).all()

print("\nDataset loading: PASSED")

URL TRAINING DATA
Shape: (164043, 2)
Phishing rate: 42.46%

URL VALIDATION DATA
Shape: (34942, 2)
Phishing rate: 42.10%

Dataset loading: PASSED


In [50]:
import time

# I extract features from the training URLs.
start_time = time.perf_counter()

X_url_train = extract_features_dataframe(
    url_train_df["url"]
)

train_feature_time = time.perf_counter() - start_time

print("Training feature extraction completed.")
print("Shape:", X_url_train.shape)
print(f"Time: {train_feature_time:.2f} seconds")


# I extract features from the validation URLs.
start_time = time.perf_counter()

X_url_val = extract_features_dataframe(
    url_val_df["url"]
)

val_feature_time = time.perf_counter() - start_time

print("\nValidation feature extraction completed.")
print("Shape:", X_url_val.shape)
print(f"Time: {val_feature_time:.2f} seconds")


# I retain the original labels separately.
y_url_train = url_train_df["label"].reset_index(drop=True)
y_url_val = url_val_df["label"].reset_index(drop=True)

Training feature extraction completed.
Shape: (164043, 21)
Time: 13.03 seconds

Validation feature extraction completed.
Shape: (34942, 21)
Time: 2.82 seconds


In [52]:
# I verify the feature matrices before modeling.
assert X_url_train.shape == (len(url_train_df), 21)
assert X_url_val.shape == (len(url_val_df), 21)

# I ensure both partitions use identical feature columns.
assert X_url_train.columns.equals(X_url_val.columns)

# I verify that every extracted feature is numerical.
assert all(
    pd.api.types.is_numeric_dtype(dtype)
    for dtype in X_url_train.dtypes
)

# I check for missing and infinite numerical values.
assert np.isfinite(X_url_train.to_numpy()).all()
assert np.isfinite(X_url_val.to_numpy()).all()

# I verify feature-label alignment.
assert len(X_url_train) == len(y_url_train)
assert len(X_url_val) == len(y_url_val)

print("URL FEATURE MATRIX VALIDATION")
print("Training shape:", X_url_train.shape)
print("Validation shape:", X_url_val.shape)
print("Feature columns:", X_url_train.shape[1])
print("Missing/infinite values: NONE")
print("Feature alignment: PASSED")
print("Overall validation: PASSED")

URL FEATURE MATRIX VALIDATION
Training shape: (164043, 21)
Validation shape: (34942, 21)
Feature columns: 21
Missing/infinite values: NONE
Feature alignment: PASSED
Overall validation: PASSED


## 9. URL Classification Baseline

I establish a majority-class baseline before training my URL phishing classifier.

The training dataset contains approximately 57.54% legitimate URLs and 42.46% phishing URLs.

A classifier that always predicts the majority class can achieve reasonable accuracy without identifying any phishing URLs.

I use this baseline to demonstrate why accuracy alone is insufficient for evaluating URL detection.

I also define a reusable evaluation function to ensure consistent metric calculations across subsequent experiments.

In [55]:
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

RANDOM_STATE = 42

print("Modeling dependencies imported successfully.")

Modeling dependencies imported successfully.


In [59]:
# I define a consistent evaluation function for URL classifiers.
def evaluate_url_model(model_name, y_true, y_pred):

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        ),
    }

    print(f"\n{model_name.upper()} RESULTS")

    for metric, value in results.items():
        if metric == "Model":
            print(f"{metric}: {value}")
        else:
            print(f"{metric}: {value:.4f}")

    print("\nCONFUSION MATRIX")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

    print("\nCLASSIFICATION REPORT")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["Legitimate", "Phishing"],
            zero_division=0,
        )
    )

    return results


print("Evaluation function ready.")

Evaluation function ready.


In [61]:
# I train a majority-class classifier without learning URL patterns.
url_baseline = DummyClassifier(
    strategy="most_frequent"
)

url_baseline.fit(
    X_url_train,
    y_url_train
)

# I generate predictions on the validation partition.
url_baseline_predictions = url_baseline.predict(
    X_url_val
)

# I evaluate the baseline.
url_baseline_results = evaluate_url_model(
    "Majority Class",
    y_url_val,
    url_baseline_predictions
)


MAJORITY CLASS RESULTS
Model: Majority Class
Accuracy: 0.5790
Precision: 0.0000
Recall: 0.0000
F1: 0.0000

CONFUSION MATRIX
[[20233     0]
 [14709     0]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.58      1.00      0.73     20233
    Phishing       0.00      0.00      0.00     14709

    accuracy                           0.58     34942
   macro avg       0.29      0.50      0.37     34942
weighted avg       0.34      0.58      0.42     34942



## 10. URL Classification — Logistic Regression

I train my first URL phishing classifier using Logistic Regression and the 21 numerical features extracted from URL strings.

Because the features have different numerical scales, I standardize them using StandardScaler.

I place the scaler and classifier inside a scikit-learn Pipeline to ensure that feature normalization is fitted only on the training partition.

I begin with standard regularization and no class weighting, allowing me to establish an initial supervised learning baseline.

I evaluate phishing precision, recall, F1-score, accuracy, and the confusion matrix using the hostname-aware validation partition.

In [66]:
import time

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# I build the URL Logistic Regression pipeline.
url_logreg_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            class_weight=None,
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    )
])

# I measure the training time.
start_time = time.perf_counter()

url_logreg_pipeline.fit(
    X_url_train,
    y_url_train
)

url_logreg_training_time = (
    time.perf_counter() - start_time
)

print("URL Logistic Regression trained successfully.")
print(
    f"Training time: {url_logreg_training_time:.2f} seconds"
)
print(
    "Features used:",
    url_logreg_pipeline.n_features_in_
)

URL Logistic Regression trained successfully.
Training time: 0.78 seconds
Features used: 21


In [68]:
# I generate predictions using the trained URL classifier.
url_logreg_predictions = url_logreg_pipeline.predict(
    X_url_val
)

# I retain probability estimates for later threshold analysis.
url_logreg_probabilities = (
    url_logreg_pipeline.predict_proba(X_url_val)[:, 1]
)

print("Validation predictions generated successfully.")

print(
    "Predicted phishing URLs:",
    int(url_logreg_predictions.sum())
)

print(
    "Actual phishing URLs:",
    int(y_url_val.sum())
)

Validation predictions generated successfully.
Predicted phishing URLs: 14473
Actual phishing URLs: 14709


In [70]:
# I evaluate URL Logistic Regression on the validation partition.
url_logreg_results = evaluate_url_model(
    "Logistic Regression",
    y_url_val,
    url_logreg_predictions
)


LOGISTIC REGRESSION RESULTS
Model: Logistic Regression
Accuracy: 0.9928
Precision: 0.9994
Recall: 0.9834
F1: 0.9914

CONFUSION MATRIX
[[20225     8]
 [  244 14465]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.99      1.00      0.99     20233
    Phishing       1.00      0.98      0.99     14709

    accuracy                           0.99     34942
   macro avg       0.99      0.99      0.99     34942
weighted avg       0.99      0.99      0.99     34942



## 11. URL Feature Influence and Dataset Bias Analysis

My Logistic Regression classifier achieved unusually strong validation performance using only 21 lexical and structural URL features.

Before selecting the model, I investigate which features contribute most strongly to its predictions.

I examine the standardized Logistic Regression coefficients and compare feature distributions between legitimate and phishing URLs.

I also measure the predictive strength of individual features to identify potential dataset-specific patterns.

This analysis helps me distinguish promising classification performance from possible collection biases or shortcuts that may limit generalization to unfamiliar URLs.

In [73]:
# I extract the trained Logistic Regression coefficients.
classifier = url_logreg_pipeline.named_steps["classifier"]

feature_importance = pd.DataFrame({
    "feature": X_url_train.columns,
    "coefficient": classifier.coef_[0]
})

# I calculate coefficient magnitudes for ranking.
feature_importance["absolute_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance = (
    feature_importance
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

print("TOP 15 LOGISTIC REGRESSION FEATURES")

display(feature_importance.head(15).round(4))

TOP 15 LOGISTIC REGRESSION FEATURES


,feature,coefficient,absolute_coefficient
0,slash_count,11.1045,11.1045
1,is_https,-6.0223,6.0223
2,special_char_count,2.4734,2.4734
3,digit_ratio,1.7104,1.7104
4,hostname_length,1.2817,1.2817
5,path_depth,0.6886,0.6886
6,path_length,0.5734,0.5734
7,suspicious_keyword_count,0.5398,0.5398
8,hyphen_count,0.4760,0.4760
9,query_length,-0.3383,0.3383


In [75]:
# I combine the training features and labels for analysis.
url_feature_analysis = X_url_train.copy()

url_feature_analysis["label"] = y_url_train.to_numpy()

# I select structural features for class comparison.
selected_features = [
    "url_length",
    "hostname_length",
    "path_length",
    "digit_ratio",
    "subdomain_count",
    "is_https",
    "is_ip_host",
    "suspicious_keyword_count"
]

# I compare average feature values by classification label.
class_feature_summary = (
    url_feature_analysis
    .groupby("label")[selected_features]
    .mean()
    .T
)

class_feature_summary.columns = [
    "Legitimate",
    "Phishing"
]

print("FEATURE DISTRIBUTIONS BY CLASS")

display(class_feature_summary.round(4))

FEATURE DISTRIBUTIONS BY CLASS


,Legitimate,Phishing
url_length,27.2286,46.1463
hostname_length,19.2286,24.6459
path_length,0.0000,8.1727
digit_ratio,0.0020,0.0625
subdomain_count,1.1622,1.1698
is_https,1.0000,0.4805
is_ip_host,0.0000,0.0072
suspicious_keyword_count,0.0033,0.0872


In [77]:
from sklearn.metrics import roc_auc_score

# I measure each feature's individual class-separation ability.
single_feature_results = []

for feature in X_url_train.columns:

    values = X_url_train[feature]

    # I skip features that have no variation.
    if values.nunique() <= 1:
        continue

    auc = roc_auc_score(
        y_url_train,
        values
    )

    # I account for features whose direction is reversed.
    separation_auc = max(auc, 1 - auc)

    single_feature_results.append({
        "feature": feature,
        "auc": auc,
        "separation_auc": separation_auc
    })

single_feature_summary = pd.DataFrame(
    single_feature_results
).sort_values(
    "separation_auc",
    ascending=False
)

print("INDIVIDUAL FEATURE SEPARATION")

display(
    single_feature_summary
    .head(15)
    .round(4)
)

INDIVIDUAL FEATURE SEPARATION


,feature,auc,separation_auc
9,slash_count,0.7965,0.7965
2,path_length,0.7965,0.7965
6,special_char_count,0.7891,0.7891
17,is_https,0.2402,0.7598
4,digit_count,0.7258,0.7258
5,digit_ratio,0.7241,0.7241
0,url_length,0.7186,0.7186
8,hyphen_count,0.6512,0.6512
15,path_depth,0.6321,0.6321
1,hostname_length,0.6318,0.6318


## 12. Dataset Shortcut Investigation

My feature analysis revealed substantial structural differences between legitimate and phishing URLs.

In the training dataset, legitimate URLs consistently use HTTPS and have empty paths, while phishing URLs exhibit more varied structures.

These differences may allow machine learning models to distinguish dataset collection patterns rather than learning generalizable phishing characteristics.

I investigate this possibility using simple rule-based predictions and examine the distribution of HTTPS usage and URL paths across the training and validation partitions.

I use this diagnostic to assess the credibility of my current validation results before proceeding with additional model development.

In [80]:
# I examine potentially biased features across both partitions.

def inspect_url_patterns(X, y, partition_name):

    analysis = X[
        ["is_https", "path_length", "slash_count"]
    ].copy()

    analysis["label"] = y.to_numpy()

    print(f"\n{partition_name.upper()} — URL PATTERNS")

    summary = (
        analysis.groupby("label")
        .agg(
            records=("label", "size"),
            https_rate=("is_https", "mean"),
            empty_path_rate=(
                "path_length",
                lambda values: (values == 0).mean()
            ),
            average_slashes=("slash_count", "mean")
        )
    )

    display(summary.round(4))


inspect_url_patterns(
    X_url_train,
    y_url_train,
    "Training"
)

inspect_url_patterns(
    X_url_val,
    y_url_val,
    "Validation"
)


TRAINING — URL PATTERNS


,records,https_rate,empty_path_rate,average_slashes
label,,,,
0,94392,1.0000,1.000,2.0000
1,69651,0.4805,0.407,3.0024



VALIDATION — URL PATTERNS


,records,https_rate,empty_path_rate,average_slashes
label,,,,
0,20233,1.0000,1.0000,2.0000
1,14709,0.4705,0.4165,2.9399


In [82]:
# I create simple classifiers using potentially biased features.

rule_predictions = {
    "HTTP Only": (
        X_url_val["is_https"] == 0
    ).astype(int),

    "Non-Empty Path Only": (
        X_url_val["path_length"] > 0
    ).astype(int),

    "HTTP or Non-Empty Path": (
        (X_url_val["is_https"] == 0)
        | (X_url_val["path_length"] > 0)
    ).astype(int)
}

rule_results = []

for name, predictions in rule_predictions.items():

    rule_results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            y_url_val, predictions
        ),
        "Precision": precision_score(
            y_url_val, predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_url_val, predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_url_val, predictions,
            zero_division=0
        )
    })

# I include Logistic Regression as the reference model.
rule_results.append(url_logreg_results)

rule_comparison = pd.DataFrame(
    rule_results
).sort_values(
    "F1",
    ascending=False
)

print("RULE-BASED CLASSIFIER COMPARISON")

display(rule_comparison.round(4))

RULE-BASED CLASSIFIER COMPARISON


,Model,Accuracy,Precision,Recall,F1
3,Logistic Regression,0.9928,0.9994,0.9834,0.9914
2,HTTP or Non-Empty Path,0.9895,1.0000,0.9750,0.9873
1,Non-Empty Path Only,0.8247,1.0000,0.5835,0.7369
0,HTTP Only,0.8019,1.0000,0.5295,0.6924


In [84]:
# I isolate URLs that resemble legitimate training examples
# in terms of HTTPS usage and path structure.

difficult_mask = (
    (X_url_val["is_https"] == 1)
    & (X_url_val["path_length"] == 0)
)

X_difficult = X_url_val.loc[difficult_mask]
y_difficult = y_url_val.loc[difficult_mask]

difficult_predictions = url_logreg_predictions[
    difficult_mask.to_numpy()
]

print("HTTPS ROOT-URL SUBSET")

print("Total URLs:", len(X_difficult))

print("\nClass distribution:")
print(y_difficult.value_counts().sort_index())

print("\nPhishing proportion:")
print(f"{y_difficult.mean():.2%}")

if y_difficult.nunique() == 2:

    difficult_results = evaluate_url_model(
        "Logistic Regression — HTTPS Root URLs",
        y_difficult,
        difficult_predictions
    )

else:
    print(
        "\nOnly one class is represented. "
        "Binary performance metrics cannot be "
        "meaningfully compared on this subset."
    )

HTTPS ROOT-URL SUBSET
Total URLs: 20601

Class distribution:
label
0    20233
1      368
Name: count, dtype: int64

Phishing proportion:
1.79%

LOGISTIC REGRESSION — HTTPS ROOT URLS RESULTS
Model: Logistic Regression — HTTPS Root URLs
Accuracy: 0.9878
Precision: 0.9394
Recall: 0.3370
F1: 0.4960

CONFUSION MATRIX
[[20225     8]
 [  244   124]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.99      1.00      0.99     20233
    Phishing       0.94      0.34      0.50       368

    accuracy                           0.99     20601
   macro avg       0.96      0.67      0.74     20601
weighted avg       0.99      0.99      0.98     20601



## 13. Dataset Bias Investigation — Findings

My initial URL classifier achieved 99.28% validation accuracy and a phishing F1-score of 99.14%.

However, further investigation revealed substantial structural differences between the dataset's legitimate and phishing URLs.

All legitimate URLs in the training and validation partitions used HTTPS and had empty paths. In contrast, phishing URLs exhibited more varied URL structures.

To investigate the impact of these patterns, I evaluated a simple rule that classified URLs as phishing when they used HTTP or contained a non-empty path.

This rule achieved a validation F1-score of 98.73% without any model training.

I then evaluated Logistic Regression on the subset of HTTPS URLs with empty paths.

Although the classifier maintained high accuracy on this imbalanced subset, phishing recall decreased to 33.70%, with 244 of 368 phishing URLs missed.

These findings indicate that the dataset contains strong structural shortcuts that may inflate model performance.

I therefore retain the existing Logistic Regression model as a benchmark but do not consider its validation metrics sufficient evidence of reliable real-world phishing detection.

Before finalizing the URL detector, I will investigate additional data sources containing more diverse legitimate and phishing URL structures.

In [87]:
import json

# I preserve the dataset-bias diagnostic results.
url_bias_results = {
    "dataset": "PhiUSIIL",
    "validation_urls": 34942,
    "logistic_regression": {
        "accuracy": url_logreg_results["Accuracy"],
        "precision": url_logreg_results["Precision"],
        "recall": url_logreg_results["Recall"],
        "f1": url_logreg_results["F1"]
    },
    "simple_rule": {
        "description": "HTTP or non-empty URL path",
        "accuracy": 0.9895,
        "precision": 1.0,
        "recall": 0.9750,
        "f1": 0.9873
    },
    "https_root_subset": {
        "total": 20601,
        "legitimate": 20233,
        "phishing": 368,
        "true_positives": 124,
        "false_negatives": 244,
        "false_positives": 8,
        "true_negatives": 20225
    },
    "conclusion": (
        "Strong structural collection bias identified. "
        "Independent evaluation is needed before deployment claims."
    )
}

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

bias_report_path = OUTPUTS_DIR / "url_dataset_bias_report.json"

with bias_report_path.open("w", encoding="utf-8") as file:
    json.dump(url_bias_results, file, indent=2)

print("Dataset bias report saved:", bias_report_path.name)

Dataset bias report saved: url_dataset_bias_report.json


## 14. Conclusion — PhiUSIIL Dataset Investigation

I initially developed a URL phishing classifier using the PhiUSIIL dataset and 21 lexical and structural features extracted directly from URL strings.

My Logistic Regression model achieved 99.28% validation accuracy and a phishing F1-score of 99.14%. Although these results appeared promising, I decided to investigate which features were driving the predictions before proceeding with more complex algorithms.

The investigation revealed substantial structural differences between legitimate and phishing URLs. Every legitimate URL in the training and validation partitions used HTTPS and had an empty path.

A simple rule that classified URLs as phishing whenever they used HTTP or contained a non-empty path achieved an F1-score of 98.73%, without requiring any machine learning.

I also evaluated the Logistic Regression model on HTTPS URLs with empty paths. Its phishing recall dropped to 33.70%, revealing a significant limitation that the overall validation metrics had concealed.

These findings suggest that the dataset contains structural collection biases that make its classification task considerably easier than real-world URL phishing detection.

I therefore decided not to finalize or deploy the classifier based on these results.

Instead, I retained the dataset, feature extractor, and experimental findings as a documented investigation and moved to an additional dataset to assess generalization more credibly.

The main lesson from this experiment is that strong evaluation metrics are not sufficient on their own. Understanding what a model has learned, and whether those patterns are likely to generalize, is equally important.
